# Parse HTML Docs
This notebook contains functions that:
1) Retrieves a list of fanwork links from a text file created with the functions in get_fanwork_links.ipynb
2) For each fanwork link, creates a BeautifulSoup object, parses the resulting html document, and extracts desired metadata about each fanwork
3) Saves the extracted metadata in a json structure

In [56]:
from bs4 import BeautifulSoup
import requests
import re
import json

# Retrieve fanworks list from text file

In [57]:
def get_fanwork_links(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    fanwork_links = open(filepath).readlines()
    for i in range(len(fanwork_links)):
        fanwork_links[i] = fanwork_links[i].replace("\n","")
    return fanwork_links

In [58]:
# test get_fanwork_links
fanwork_links = get_fanwork_links('../../data/fanwork_links.txt')
print(fanwork_links)

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/55118308', 'https://archiveofourown.org/works/66001534', 'https://archiveofourown.org/works/62477026', 'https://archiveofourown.org/works/62036794', 'https://archiveofourown.org/works/60847894', 'https://archiveofourown.org/works/51300544', 'https://archiveofourown.org/works/56817901', 'https://archiveofourown.org/works/56876515', 'https://archiveofourown.org/work

# Create soup objects

In [59]:
def get_html_doc_fanworks(url):
    """Takes in the url for any given Ao3 fanwork and returns an html document of that page."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [60]:
# test get_html_doc_fanworks
test_url = fanwork_links[0]
soup = get_html_doc_fanworks(test_url)
print(soup)

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
<meta content="ie=edge" http-equiv="x-ua-compatible"/>
<meta content="fanfiction, transformative works, otw, fair use, archive" name="keywords"/>
<meta content="en-US" name="language"/>
<meta content="fandom" name="subject"/>
<meta content="An Archive of Our Own, a project of the Organization for Transformative Works" name="description"/>
<meta content="GLOBAL" name="distribution"/>
<meta content="transformative works" name="classification"/>
<meta content="Organization for Transformative Works" name="author"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<meta content="nointentdetection" name="chrome"/>
<meta content="telephone=no" name="format-detection"/>
<title>The Circle of the Ghost Lamp - Chapter 1 - puck1919 - Private Nightmares | Project Ghostlight, Candela Obscura (Critical Role Web Series) [Archive of Our Own]</title>
<link href="/stylesheets/skins/skin_1_default/1_site_screen_.css" m

In [61]:
def get_all_dt(soup):
    """Takes in BeautifulSoup object from Ao3 fanwork and returns a list of all 'dt' elements of the html document. 'dt' elements represent the metadata categories present for the fanwork."""
    all_dt = soup.find_all('dt')
    all_dt = list(all_dt)
    return all_dt

In [62]:
# test get_all_dt
all_dt = get_all_dt(soup)
print(all_dt)

[<dt><label for="user_session_login_small">Username or email:</label></dt>, <dt><label for="user_session_password_small">Password:</label></dt>, <dt class="rating tags">

              Rating:
          </dt>, <dt class="warning tags">
<a href="/tos_faq#tags">Archive Warning</a>:
          </dt>, <dt class="category tags">

              Category:
          </dt>, <dt class="fandom tags">

              Fandoms:
          </dt>, <dt class="relationship tags">

              Relationships:
          </dt>, <dt class="character tags">

              Characters:
          </dt>, <dt class="freeform tags">

              Additional Tags:
          </dt>, <dt class="language">
        Language:
      </dt>, <dt class="stats">Stats:</dt>, <dt class="published">Published:</dt>, <dt class="status">Updated:</dt>, <dt class="words">Words:</dt>, <dt class="chapters">Chapters:</dt>, <dt class="comments">Comments:</dt>, <dt class="kudos">Kudos:</dt>, <dt class="bookmarks">Bookmarks:</dt>, <dt class

In [63]:
def update_all_dt(dt_list):
    dt_dict = {
        '<dt><label for="user_session_login_small">Username or email:</label></dt>':'user_session_login',
        '<dt><label for="user_session_password_small">Password:</label></dt>':'user_session_password',
        '<dt class="rating tags">':'Rating',
        '<dt class="warning tags">':'Archive Warning',
        '<dt class="category tags">':'Categories',
        '<dt class="fandom tags">':'Fandoms',
        '<dt class="relationship tags">':'Relationships',
        '<dt class="character tags">':'Characters',
        '<dt class="freeform tags">':'Additional Tags',
        '<dt class="language">':'Language',
        '<dt class="stats">':'Stats',
        '<dt class="published">':'Date Published',
        '<dt class="status">':'Date Updated',
        '<dt class="words">':'Word Count',
        '<dt class="chapters">':'Chapters',
        '<dt class="comments">':'Comments',
        '<dt class="kudos">':'Kudos',
        '<dt class="bookmarks">':'Bookmarks',
        '<dt class="hits">':'Hits',
        '<dt class="landmark">Note:</dt>':'note',
        '<dt><label for="comment_name_for_211688701">Guest name</label></dt>':'guest_commenter_name',
        '<dt><label for="comment_email_for_211688701">Guest email</label></dt>':'guest_commenter_email'
    }

    for i in range(len(dt_list)):
        dt_list[i] = str(dt_list[i])

    for key in dt_dict.keys():
        for i in range(len(dt_list)):
            key_string = str(key)
            if key_string in dt_list[i]:
                dt_list[i] = dt_dict[key]
            else:
                continue
    return dt_list

In [64]:
print(type(all_dt[2]))

<class 'bs4.element.Tag'>


In [65]:
all_dt_updated = update_all_dt(all_dt)
print(all_dt_updated)

['user_session_login', 'user_session_password', 'Rating', 'Archive Warning', 'Categories', 'Fandoms', 'Relationships', 'Characters', 'Additional Tags', 'Language', 'Stats', 'Date Published', 'Date Updated', 'Word Count', 'Chapters', 'Comments', 'Kudos', 'Bookmarks', 'Hits', 'note', 'guest_commenter_name', 'guest_commenter_email']


In [66]:
print(len(all_dt))

22


In [67]:
print(all_dt_updated[4])

Categories


In [68]:
def get_all_dd(soup):
    """Takes in BeautifulSoup object from Aoe fanwork and returns a list of all 'dd' elements of the html document. 'dd' elements represent the metadata values present in the fanwork."""
    all_dd = soup.find_all('dd')
    all_dd = list(all_dd)

    # convert dd_list items to strings
    for i in range(len(all_dd)):
        all_dd[i] = str(all_dd[i])

    return all_dd

In [386]:
# test get_all_dd
all_dd = get_all_dd(soup)

# Parse html docs and clean metadata

## strip_dd

In [71]:
def strip_dd(dd_list, text_to_strip):
    for item in text_to_strip:
        for i in range(len(dd_list)):
            if item in dd_list[i]:
                dd_list[i] = dd_list[i].replace(item, "")
            else:
                continue
    return dd_list

In [251]:
print(all_dd[0])

<dd><input autocomplete="on" id="user_session_login_small" name="user[login]" type="text"/></dd>


In [387]:
text_to_strip = ['\n',
                 '<ul class="commas">',
                 '<li>','</li>',
                 '</ul>',
                 '<a class="tag" href="','</a>',
                 '<dd class="rating tags">',
                 '/tags/Mature/works">',
                 '<dd class="warning tags">',
                 '/tags/Graphic%20Depictions%20Of%20Violence/works">',
                 '<dd class="category tags">',
                 '/tags/Gen/works">',
                 '</dd>',
                 '<dd class="language" lang="en">',
                 '<!-- end of cache -->',
                 '<dd class="stats">',
                 '<dl class="stats">',
                 '<dt class="published">',
                 '</dt><dd class="published">',
                 '<dt class="status">',
                 '</dt><dd class="status">',
                 '<dt class="words">',
                 '</dt><dd class="words">',
                 '<dt class="chapters">',
                 '</dt><dd class="chapters">',
                 '<dt class="comments">',
                 '</dt><dd class="comments">',
                 '<dt class="kudos">',
                 '</dt><dd class="kudos">',
                 '<dt class="bookmarks">',
                 '</dt><dd class="bookmarks">',
                 'bookmarks">',
                 '<a href="/',
                 '<dt class="hits">',
                 '</dt><dd class="hits">',
                 '</dl>',
                 '<dd class="instructions comment_form">',
                 '<dd><input id="',
                 '" name="comment[name]" type="text"/><script>',
                 '/<![CDATA[var ',
                 ' = new LiveValidation',
                 ', { wait: 500, onlyOnBlur: false }',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your name.","validMessage":""})',
                 ';//]]>',
                 '</script>',
                 '<input id="',
                 '" name="comment[email]"',
                 ' type="text"/>',
                 '<script>',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your email address.","validMessage":""})',
                 '<dd><input autocomplete="on" ',
                 'id="user_session_login_small" '
                 ]
all_dd_stripped = strip_dd(all_dd, text_to_strip)

## cleanup_rating - maybe can get rid of

In [258]:
def cleanup_rating(dd_list):
    ratings_dict = {
        '"ratingtags"/tags/Not%20Rated">Not Rated':'Not Rated',
        '"ratingtags"/tags/General%20Audiences">General Audiences':'General Audiences',
        '"ratingtags"/tags/Teen%20And%20Up%20Audiences">Teen And Up Audiences':'Teen',
        '"ratingtags"/tags/Mature/works">Mature':'Mature',
        '"ratingtags"/tags/Explicit/works">Explicit':'Explicit'
                    }

    for key in ratings_dict.keys():
        for i in range(len(dd_list)):
            if key == dd_list[i]:
                dd_list[i] = ratings_dict[key]
            else:
                dd_list[i] = dd_list[i]
    return dd_list

In [259]:
# test cleanup_rating
dd_list1 = cleanup_rating(all_dd_stripped1)
print(dd_list1[2])

/tags/Mature/worksMature


## cleanup_warnings - maybe can get rid of

In [ ]:
def cleanup_warnings(dd_list):
    warnings_dict = {
        '"warningtags"/tags/Graphic%20Depictions%20Of%20Violence/works">GraphicDepictionsOfViolence':'Graphic Depictions of Violence',
        '"warningtags"/tags/Choose%20Not%20To%20Use%20Archive%20Warnings">Creator Chose Not To Use Archive Warnings':'Creator Chose Not To Use Archive Warnings',
        '"warningtags"/tags/Major%20Character%20Death">Major Character Death':'Major Character Death',
        '"warningtags"/tags/No%20Archive%20Warnings%20Apply">No Archive Warnings Apply':'No Archive Warnings Apply',
        '"warningtags"/tags/Rape*s*Non-Con">Rape/Non-Con':'Rape/Non-Con',
        '"warningtags"/tags/Underage%20Sex">Underage Sex':'Underage Sex'
    }

    for key in warnings_dict.keys():
        for i in range(len(dd_list)):
            if key == dd_list[i]:
                dd_list[i] = warnings_dict[key]
            else:
                dd_list[i] = dd_list[i]
    return dd_list

In [ ]:
# testing cleanup_warnings
dd_list2 = cleanup_warnings(dd_list1)
print(dd_list2[3])

## cleanup_categories - maybe can get rid of

In [ ]:
def cleanup_categories(dd_list):
    categories_dict = {
        '"categorytags"/tags/Gen/works">Gen':'Gen',
        '"categorytags"/tags/F*s*F">F/F':'F/F',
        '"categorytags"/tags/F*s*M">F/M':'F/M',
        '"categorytags"/tags/M*s*M">M/M':'M/M',
        '"categorytags"/tags/Multi">Multi':'Multi',
        '"categorytags"/tags/Other">Other':'Other'
    }

    for key in categories_dict.keys():
        for i in range(len(dd_list)):
            if key == dd_list[i]:
                dd_list[i] = categories_dict[key]
            else:
                dd_list[i] = dd_list[i]
    return dd_list

In [ ]:
# testing cleanup_categories
dd_list3 = cleanup_categories(dd_list2)
print(dd_list3[4])

## cleanup_fandom_tags

In [425]:
print(all_dd_stripped[5])

<dd class="fandom tags">/tags/Private%20Nightmares%20%7C%20Project%20Ghostlight/works">Private Nightmares | Project Ghostlight/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works">Candela Obscura (Critical Role Web Series)


In [426]:
def cleanup_fandom_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="fandom tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="fandom tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [429]:
# test cleanup_fandom_tags
all_dd_stripped = cleanup_fandom_tags(all_dd_stripped)

In [430]:
print(all_dd_cleaned_fandoms[5])

['', 'Private Nightmares | Project Ghostlight', 'Candela Obscura (Critical Role Web Series)']


## cleanup_relationship_tags

In [388]:
print(all_dd_stripped[6])

<dd class="relationship tags">/tags/Eddie%20*a*%20Katie%20(Private%20Nightmares)/works">Eddie &amp; Katie (Private Nightmares)/tags/Naomi%20*a*%20Jade%20(Private%20Nightmares)/works">Naomi &amp; Jade (Private Nightmares)/tags/Eddie%20*a*%20Jade%20(Private%20Nightmares)/works">Eddie &amp; Jade (Private Nightmares)/tags/Gable%20*a*%20Jade%20(Private%20Nightmares)/works">Gable &amp; Jade (Private Nightmares)


In [389]:
def cleanup_relationship_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="relationship tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="relationship tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [431]:
# test cleanup_relationship_tags
all_dd_stripped = cleanup_relationship_tags(all_dd_stripped)

In [432]:
print(all_dd_stripped[6])

['', 'Eddie & Katie (Private Nightmares)', 'Naomi & Jade (Private Nightmares)', 'Eddie & Jade (Private Nightmares)', 'Gable & Jade (Private Nightmares)']


## cleanup_character_tags

In [392]:
print(all_dd_stripped[7])

<dd class="character tags">/tags/Katie%20(Private%20Nightmares)/works">Katie (Private Nightmares)/tags/Gable%20(Private%20Nightmares)/works">Gable (Private Nightmares)/tags/Jade%20(Private%20Nightmares)/works">Jade (Private Nightmares)/tags/Eddie%20(Private%20Nightmares)/works">Eddie (Private Nightmares)/tags/Naomi%20(Private%20Nightmares)/works">Naomi (Private Nightmares)/tags/Robert%20(Private%20Nightmares)/works">Robert (Private Nightmares)/tags/Malcom%20Trills%20(Candela%20Obscura)/works">Malcom Trills (Candela Obscura)/tags/Edgar%20Lycoris/works">Edgar Lycoris/tags/Horatio%20(Private%20Nightmares)/works">Horatio (Private Nightmares)/tags/Vulo%20(Private%20Nightmares)/works">Vulo (Private Nightmares)/tags/Manny%20(Private%20Nightmares)/works">Manny (Private Nightmares)/tags/Cosmo%20Grimm%20(Candela%20Obscura)/works">Cosmo Grimm (Candela Obscura)


In [398]:
def cleanup_character_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="character tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="character tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [433]:
# test cleanup_relationship_tags
all_dd_stripped = cleanup_character_tags(all_dd_stripped)

In [434]:
print(all_dd_stripped[7])

['', 'Katie (Private Nightmares)', 'Gable (Private Nightmares)', 'Jade (Private Nightmares)', 'Eddie (Private Nightmares)', 'Naomi (Private Nightmares)', 'Robert (Private Nightmares)', 'Malcom Trills (Candela Obscura)', 'Edgar Lycoris', 'Horatio (Private Nightmares)', 'Vulo (Private Nightmares)', 'Manny (Private Nightmares)', 'Cosmo Grimm (Candela Obscura)']


## cleanup_additional_tags

In [394]:
print(all_dd_stripped[8])

<dd class="freeform tags">/tags/Vampires/works">Vampires/tags/Gaslamp%20Fantasy/works">Gaslamp Fantasy/tags/Horror/works">Horror/tags/Monster%20-%20Freeform/works">Monster - Freeform/tags/Blood/works">Blood/tags/Body%20Horror/works">Body Horror/tags/Autopsy/works">Autopsy/tags/Mystery/works">Mystery/tags/don't%20look%20too%20hard%20at%20the%20Candela%20timeline/works">don't look too hard at the Candela timeline


In [401]:
def cleanup_additional_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="freeform tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="freeform tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [435]:
# test cleanup_additional_tags
all_dd_stripped = cleanup_additional_tags(all_dd_stripped)

In [436]:
print(all_dd_stripped[8])

['', 'Vampires', 'Gaslamp Fantasy', 'Horror', 'Monster - Freeform', 'Blood', 'Body Horror', 'Autopsy', 'Mystery', "don't look too hard at the Candela timeline"]


## multiple_tags_to_lists - can delete

In [ ]:
def multiple_tags_to_lists(dd_list):
    for i in range(len(dd_list)):
        if '/tags' in dd_list[i]:
            dd_list[i] = dd_list[i].split('/tags')
        elif '"relationshiptags"' in dd_list[i]:
            dd_list[i] = dd_list[i].split('">')
            dd_list[i] = dd_list[i][1]
        else:
            dd_list[i] = dd_list[i]
    return dd_list

# so we have fandoms, relationships, characters, and additional tags
# all of which require the regex thing

In [ ]:
dd_list4 = multiple_tags_to_lists(dd_list3)

In [ ]:
print(dd_list4[8])

## get_idx_multiple_tags - can delete

In [ ]:
def get_idx_multiple_tags(dd_list):
    indices = []
    for i in range(len(dd_list)):
        if type(dd_list[i]) == list:
            indices.append(i)
        else:
            continue
    del indices[4:9] # remove last index because it's a string
    return indices

In [ ]:
indices = get_idx_multiple_tags(dd_list4)
print(indices)

## cleanup_multiple_tags - can delete

In [ ]:
print(dd_list4[6][1])

In [ ]:
pattern = re.compile(r"(?<=>)(?!.*>).*")
print(pattern)
print(type(pattern))

In [ ]:
def cleanup_multiple_tags(dd_list, indices_list):
    """Uses regex to clean up metadata categories that have been converted to lists in multiple_tags_to_lists."""
    pattern = re.compile(r"(?<=>)(?!.*>).*")

    for idx in indices_list:
        current_list = dd_list[idx]
        for i in range(len(current_list)):
            match = re.search(pattern, current_list[i])
            if match:
                current_list[i] = match.group(0)
            else:
                current_list[i] = current_list[i]
            current_list[i] = current_list[i].replace("&amp;","&")
        dd_list[idx] = current_list
    return dd_list

In [ ]:
dd_list5 = cleanup_multiple_tags(dd_list4, indices)
print(dd_list5[5])
print(dd_list5[6])
print(dd_list5[7])
print(dd_list5[8])

# Retrieve Metadata

## get_rating

In [409]:
def get_rating(dt_list, dd_list):
    # find ratings index
    if 'Rating' in dt_list:
        index = dt_list.index('Rating')
        rating = dd_list[index]
        return rating

    else:
        rating = 'Unknown'
        return rating

In [410]:
# test get_ratings
get_rating(all_dt_updated, all_dd_stripped)

'Mature'

## get_warnings

In [411]:
def get_warnings(dt_list, dd_list):
    # find warning index
    if 'Archive Warning' in dt_list:
        index = dt_list.index('Archive Warning')
        warning = dd_list[index]
        return warning
    else:
        warning = 'No warnings listed'
        return warning

In [412]:
# test get_warnings
get_warnings(all_dt_updated, all_dd_stripped)

'Graphic Depictions Of Violence'

## get_category

In [413]:
def get_category(dt_list, dd_list):
    # find category index
    if 'Categories' in dt_list:
        index = dt_list.index('Categories')
        category = dd_list[index]
        return category
    else:
        category = 'Unknown'
        return category

In [414]:
# test get_category
get_category(all_dt_updated, all_dd_stripped)

'Gen'

## get_language

In [415]:
print(all_dd_stripped[9])

        English      


In [420]:
def get_language(dt_list, dd_list):
    """Uses language index found in find_language_index to get language metadata value for an Ao3 work."""
    if 'Language' in dt_list:
        index = dt_list.index("Language")
        language_str = dd_list[index]
        language_str = language_str.strip()
        return language_str
    else:
        language_str = 'Unknown'
        return language_str

In [422]:
# test get_language
language = get_language(all_dt_updated, all_dd_stripped)
print(language)

English


## get_fandom

In [423]:
def get_fandom(dt_list, dd_list):
    # find fandom index
    if 'Fandoms' in dt_list:
        index = dt_list.index("Fandoms")
        fandoms = dd_list[index]
        return fandoms
    else:
        fandoms = 'No fandoms listed'
        return fandoms

In [437]:
# test get_fandom
get_fandom(all_dt_updated, all_dd_stripped)

['',
 'Private Nightmares | Project Ghostlight',
 'Candela Obscura (Critical Role Web Series)']

## get_relationships

In [438]:
def get_relationships(dt_list, dd_list):
    # find relationship index
    if 'Relationships' in dt_list:
        index = dt_list.index("Relationships")
        relationships = dd_list[index]
        return relationships
    else:
        relationships = 'No relationships listed'
        return relationships

In [439]:
# test get_relationships
get_relationships(all_dt_updated, all_dd_stripped)

['',
 'Eddie & Katie (Private Nightmares)',
 'Naomi & Jade (Private Nightmares)',
 'Eddie & Jade (Private Nightmares)',
 'Gable & Jade (Private Nightmares)']

## get_characters

In [440]:
def get_characters(dt_list, dd_list):
    # find relationship index
    if 'Characters' in dt_list:
        index = dt_list.index('Characters')
        characters = dd_list[index]
        return characters
    else:
        characters = 'No characters listed'
        return characters

In [441]:
# test get_characters
get_characters(all_dt_updated, all_dd_stripped)

['',
 'Katie (Private Nightmares)',
 'Gable (Private Nightmares)',
 'Jade (Private Nightmares)',
 'Eddie (Private Nightmares)',
 'Naomi (Private Nightmares)',
 'Robert (Private Nightmares)',
 'Malcom Trills (Candela Obscura)',
 'Edgar Lycoris',
 'Horatio (Private Nightmares)',
 'Vulo (Private Nightmares)',
 'Manny (Private Nightmares)',
 'Cosmo Grimm (Candela Obscura)']

## get_additional_tags

In [442]:
def get_additional_tags(dt_list, dd_list):
    # find additional tags index
    if 'Additional Tags' in dt_list:
        index = dt_list.index("Additional Tags")
        additional_tags = dd_list[index]
        return additional_tags
    else:
        additional_tags = 'No additional tags'
        return additional_tags

In [443]:
# test get_additional_tags
get_additional_tags(all_dt_updated, all_dd_stripped)

['',
 'Vampires',
 'Gaslamp Fantasy',
 'Horror',
 'Monster - Freeform',
 'Blood',
 'Body Horror',
 'Autopsy',
 'Mystery',
 "don't look too hard at the Candela timeline"]

## is_series

In [444]:
def is_series(dt_list):
    for i in range(len(dt_list)):
        current_string = dt_list[i]
        if 'Series' in current_string:
            is_part_of_series = True
            return is_part_of_series
        else:
            is_part_of_series = False
            return is_part_of_series

In [446]:
is_series(all_dd_stripped)

False

## get_date_published

In [448]:
print(all_dd_stripped[11])

<dd class="published">2026-03-04


In [451]:
def get_date_published(dt_list, dd_list):
    # find date published index
    if 'Date Published' in dt_list:
        index = dt_list.index('Date Published')
        date_published = dd_list[index]
        date_published = date_published.split('>')
        date_published = date_published[1]
        return date_published

    else:
        date_published = 'Unknown'
        return date_published

In [453]:
# test get_date_published
date_published = get_date_published(all_dt_updated, all_dd_stripped)
print(date_published)
print(type(date_published))

2026-03-04
<class 'str'>


## get_date_updated

In [454]:
def get_date_updated(dt_list, dd_list):
    # find date updated index
    if 'Date Updated' in dt_list:
        index = dt_list.index("Date Updated")
        date_updated = dd_list[index]
        date_updated = date_updated.split('>')
        date_updated = date_updated[1]
        return date_updated
    elif 'Date Published' in dt_list:
        index = dt_list.index('Date Published')
        date_updated = dd_list[index]
        date_updated = date_updated.split('>')
        date_updated = date_updated[1]
        return date_updated
    else:
        date_updated = 'Unknown'
        return date_updated

In [455]:
# test get_date_updated
date_updated = get_date_updated(all_dt_updated, all_dd_stripped)
print(date_updated)
print(type(date_updated))

2026-05-12
<class 'str'>


## get_word_count

In [462]:
def get_word_count(dt_list, dd_list):
    # find word count index
    if 'Word Count' in dt_list:
        index = dt_list.index("Word Count")
        word_count = dd_list[index]
        word_count = word_count.replace(',','')
        word_count = word_count.split('>')
        word_count = int(word_count[1])
        return word_count
    else:
        word_count = 'Unknown'
        return word_count

In [463]:
# test get_word_count
word_count = get_word_count(all_dt_updated, all_dd_stripped)
print(word_count)

14223


## get_chapter_count

In [464]:
def get_chapter_count(dt_list, dd_list):
    # find Chapter index
    if 'Chapters' in dt_list:
        index = dt_list.index("Chapters")
        chapter_count = dd_list[index]
        chapter_count = chapter_count.split('>')
        chapter_count = chapter_count[1]
        return chapter_count
    else:
        chapter_count = str(1)
        return chapter_count

In [465]:
# test get_chapter_count
chapter_count = get_chapter_count(all_dt_updated, all_dd_stripped)
print(chapter_count)

2/?


## get_comment_count

In [466]:
def get_comment_count(dt_list, dd_list):
    # find comment count index
    if 'Comments' in dt_list:
        index = dt_list.index("Comments")
        comment_count = dd_list[index]
        comment_count = comment_count.split('>')
        comment_count = int(comment_count[1])
        return comment_count
    else:
        comment_count = 0
        return comment_count

In [469]:
# test get_comment_count
comment_count = get_comment_count(all_dt_updated, all_dd_stripped)
print(comment_count)

4


## get_kudos_count

In [470]:
def get_kudos_count(dt_list, dd_list):
    # find kudos count index
    if 'Kudos' in dt_list:
        index = dt_list.index("Kudos")
        kudos_count = dd_list[index]
        kudos_count = kudos_count.split('>')
        kudos_count = int(kudos_count[1])
        return kudos_count
    else:
        kudos_count = 0
        return kudos_count

In [471]:
# test get_kudos_count
kudos_count = get_kudos_count(all_dt_updated, all_dd_stripped)
print(kudos_count)

6


## get_bookmarks_count

In [477]:
print(all_dd_stripped[17])

<dd class="works/80602681/2


In [478]:
def get_bookmarks_count(dt_list, dd_list):
    # find bookmarks count index
    if 'Bookmarks' in dt_list:
        index = dt_list.index("Bookmarks")
        bookmarks_count = dd_list[index]
        bookmarks_count = bookmarks_count[-1]
        bookmarks_count = int(bookmarks_count)
        return bookmarks_count

    else:
        bookmarks_count = 0
        return bookmarks_count

In [479]:
# test get_bookmarks_count
bookmarks_count = get_bookmarks_count(all_dt_updated, all_dd_stripped)
print(bookmarks_count)

2


# Create Dictionary from html items

In [480]:
def create_fanwork_dict(dt_list, dd_list):
    fanwork_dict = {}

    # add rating
    rating = get_rating(dt_list, dd_list)
    fanwork_dict['Rating'] = get_rating(dt_list, dd_list)

    # add archive warning(s)
    fanwork_dict['Archive Warnings'] = get_warnings(dt_list, dd_list)

    # category
    fanwork_dict['Category'] = get_category(dt_list, dd_list)

    # add fandom
    fanwork_dict['Fandom'] = get_fandom(dt_list, dd_list)

    # add characters
    fanwork_dict['Characters'] = get_characters(dt_list, dd_list)

    # add relationships
    fanwork_dict['Relationships'] = get_relationships(dt_list, dd_list)

    # add additional tags
    fanwork_dict['Additional Tags'] = get_additional_tags(dt_list, dd_list)

    # add language
    fanwork_dict['Language'] = get_language(dt_list, dd_list)

    # add whether it is a series
    fanwork_dict['Is Series'] = is_series(dd_list)

    # add stats
    fanwork_dict['Date Published'] = get_date_published(dt_list, dd_list)
    fanwork_dict['Date Updated'] = get_date_updated(dt_list, dd_list)
    fanwork_dict['Word Count'] = get_word_count(dt_list, dd_list)
    fanwork_dict['Chapter Count'] = get_chapter_count(dt_list, dd_list)
    fanwork_dict['Comment Count'] = get_comment_count(dt_list, dd_list)
    fanwork_dict['Kudos Count'] = get_kudos_count(dt_list, dd_list)
    fanwork_dict['Bookmarks Count'] = get_bookmarks_count(dt_list, dd_list)

    return fanwork_dict

In [481]:
# test create_fanwork_dictionary
test_dict = create_fanwork_dict(all_dt_updated, all_dd_stripped)
print(test_dict)

{'Rating': 'Mature', 'Archive Warnings': 'Graphic Depictions Of Violence', 'Category': 'Gen', 'Fandom': ['', 'Private Nightmares | Project Ghostlight', 'Candela Obscura (Critical Role Web Series)'], 'Characters': ['', 'Katie (Private Nightmares)', 'Gable (Private Nightmares)', 'Jade (Private Nightmares)', 'Eddie (Private Nightmares)', 'Naomi (Private Nightmares)', 'Robert (Private Nightmares)', 'Malcom Trills (Candela Obscura)', 'Edgar Lycoris', 'Horatio (Private Nightmares)', 'Vulo (Private Nightmares)', 'Manny (Private Nightmares)', 'Cosmo Grimm (Candela Obscura)'], 'Relationships': ['', 'Eddie & Katie (Private Nightmares)', 'Naomi & Jade (Private Nightmares)', 'Eddie & Jade (Private Nightmares)', 'Gable & Jade (Private Nightmares)'], 'Additional Tags': ['', 'Vampires', 'Gaslamp Fantasy', 'Horror', 'Monster - Freeform', 'Blood', 'Body Horror', 'Autopsy', 'Mystery', "don't look too hard at the Candela timeline"], 'Language': 'English', 'Is Series': False, 'Date Published': '2026-03-04

# Add fanworks_dict to master dictionary (all_works_dict)

In [482]:
all_works_dict = {}
print(len(all_works_dict))

0


In [485]:
def append_all_works_dict(fanwork_dict):
    current_length = len(all_works_dict)
    new_idx = current_length + 1
    all_works_dict[new_idx] = fanwork_dict
    return all_works_dict

In [486]:
# test append_all_works_dict
append_all_works_dict(test_dict)

{1: {'Rating': 'Mature',
  'Archive Warnings': 'Graphic Depictions Of Violence',
  'Category': 'Gen',
  'Fandom': ['',
   'Private Nightmares | Project Ghostlight',
   'Candela Obscura (Critical Role Web Series)'],
  'Characters': ['',
   'Katie (Private Nightmares)',
   'Gable (Private Nightmares)',
   'Jade (Private Nightmares)',
   'Eddie (Private Nightmares)',
   'Naomi (Private Nightmares)',
   'Robert (Private Nightmares)',
   'Malcom Trills (Candela Obscura)',
   'Edgar Lycoris',
   'Horatio (Private Nightmares)',
   'Vulo (Private Nightmares)',
   'Manny (Private Nightmares)',
   'Cosmo Grimm (Candela Obscura)'],
  'Relationships': ['',
   'Eddie & Katie (Private Nightmares)',
   'Naomi & Jade (Private Nightmares)',
   'Eddie & Jade (Private Nightmares)',
   'Gable & Jade (Private Nightmares)'],
  'Additional Tags': ['',
   'Vampires',
   'Gaslamp Fantasy',
   'Horror',
   'Monster - Freeform',
   'Blood',
   'Body Horror',
   'Autopsy',
   'Mystery',
   "don't look too hard at

In [487]:
print(len(all_works_dict))

1


# Convert master dictionary to json object

In [ ]:
def convert_to_json(all_works_dict):
    json_string = json.dumps(all_works_dict, indent=4)
    return json_string

In [ ]:
# test convert_to_json
test_json = convert_to_json(all_works_dict)
print(test_json)

In [ ]:
print(type(test_json))

In [ ]:
def save_json_to_file(json_string):
    with open('data/json_string.json', "w") as f:
        f.write(json_string)

In [ ]:
# test save_json_to_file
save_json_to_file(test_json)

# Test with list of fanworks

In [ ]:
# get list of fanwork links
fanwork_links = get_fanwork_links('../../data/fanwork_links.txt')
print(fanwork_links)

In [ ]:
print(len(fanwork_links))

In [ ]:
all_works_dict1 = {}
print(len(all_works_dict1))

In [ ]:
# create + fill all_works_dict
total_fanworks = len(fanwork_links)
for i in range(len(fanwork_links)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(fanwork_links[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, all_works_dict1)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict1.")
print("Process complete. all_works_dict1 populated; you may save to json.")

In [ ]:
## convert to json and save to file

# convert to json
all_candela_works_json = convert_to_json(all_works_dict1)

# save json to file
with open('../../data/all_candela_works.json', "w") as f:
    f.write(all_candela_works_json)

# Testing / Troubleshooting

## Round 1

troubleshooting note - so it seems to be working when I have a list of one link only; I'm not sure what the deal is with this

In [ ]:
def check_if_rating(all_dt):
    if 'Rating' in all_dt_updated:
        print('Rating is in all_dt_updated')
    else:
        print("nope")

In [ ]:
for i in range(len(fanwork_links)):
    fanwork_links[i] = fanwork_links[i].replace("\n","")

In [ ]:
test_fanwork_list = fanwork_links[0:10]
print(test_fanwork_list)

In [ ]:
print(test_fanwork_list[0])

In [ ]:
def check_word_count(all_dt):
    if 'Word Count' in all_dt:
        print('Word Count is in all_dt')
    else:
        print("Word count is not in all_dt")

In [ ]:
# create + fill all_works_dict
total_fanworks = len(test_fanwork_list)
all_works_dict2 = {}
for i in range(len(test_fanwork_list)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    check_word_count(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict1
    append_all_works_dict(fanwork_dict, all_works_dict2)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict2.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json = convert_to_json(all_works_dict2)

# save json to file
with open('../../data/test_candela_works.json', "w") as f:
    f.write(test_candela_works_json)

## Round 2

In [ ]:
test_fanwork_list1 = fanwork_links[0:1]
print(test_fanwork_list1)
print(type(test_fanwork_list1))

In [ ]:
# create + fill all_works_dict
total_fanworks = len(test_fanwork_list1)
all_works_dict3 = {}
for i in range(len(test_fanwork_list1)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list1[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    print(all_dt_updated[2])
    print(type(all_dt_updated[2]))
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print(fanwork_dict)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict3
    append_all_works_dict(fanwork_dict, all_works_dict3)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict3.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json1 = convert_to_json(all_works_dict3)

# save json to file
with open('../../data/test_candela_works1.json', "w") as f:
    f.write(test_candela_works_json1)

## Round 3

In [ ]:
test_fanwork_list2 = fanwork_links[1:2]
print(test_fanwork_list2)

In [ ]:
a = [10, 20, 30, 40, 50]

if 30 in a:
    print("Element exists in the list")
else:
    print("Element does not exist")

In [ ]:
# create + fill all_works_dict4
total_fanworks = len(test_fanwork_list2)
all_works_dict4 = {}
for i in range(len(test_fanwork_list2)):
    print("Current fanwork link:", str(i+1), "of", total_fanworks)

    ## load fanworks and create soup objects
    soup = get_html_doc_fanworks(test_fanwork_list2[i])
    all_dt = get_all_dt(soup)
    all_dt_updated = update_all_dt(all_dt)
    check_if_rating(all_dt_updated)
    all_dd = get_all_dd(soup)
    print("Successfully loaded fanworks and created soup objects for link", str(i+1))

    ## clean up all_dd
    text_to_strip = ['<dd>','</dd>','<ul class="commas">','/ul>','\n',' ','<li>','</li>','><aclass="tag"href="','<ddclass=','</a><']
    all_dd_stripped = strip_dd(all_dd, text_to_strip)
    dd_list1 = cleanup_rating(all_dd_stripped)
    dd_list2 = cleanup_warnings(dd_list1)
    dd_list3 = cleanup_categories(dd_list2)
    dd_list4 = multiple_tags_to_lists(dd_list3)
    indices = get_idx_multiple_tags(dd_list4)
    dd_list5 = cleanup_multiple_tags(dd_list4, indices)
    print("Successfully cleaned all_dd for link", str(i+1))

    ## create individual fanwork dictionary
    fanwork_dict = create_fanwork_dict(all_dt_updated, dd_list5)
    print("Successfully created individual fanwork dictionary for link", str(i+1))

    ## append fanwork dictionary to all_works_dict4
    append_all_works_dict(fanwork_dict, all_works_dict4)
    print("Successfully appended individual fanwork dictionary", str(i+1), "to all_works_dict4.")

In [ ]:
## convert to json and save to file
# convert to json
test_candela_works_json2 = convert_to_json(all_works_dict4)

# save json to file
with open('../../data/test_candela_works2.json', "w") as f:
    f.write(test_candela_works_json2)